# London cycle hire analysis

Exploration of TfL Santander Cycles journey data for the Data Observatory demo.

## Download the data

In [1]:
from pathlib import Path

import pandas as pd

from tfl_cycles_lib import JOURNEY_COLUMNS, download_file, load_journeys

RAW_DIR = Path("data/raw")
JOURNEYS_URL = (
    "https://cycling.data.tfl.gov.uk/usage-stats/"
    "399JourneyDataExtract01Jul2024-14Jul2024.csv"
)
BIKEPOINT_URL = "https://api.tfl.gov.uk/BikePoint"

In [2]:
journeys_path = download_file(JOURNEYS_URL, RAW_DIR)
bikepoint_path = download_file(
    BIKEPOINT_URL, RAW_DIR, filename="bikepoint.json"
)
journeys_path, bikepoint_path

(PosixPath('data/raw/399JourneyDataExtract01Jul2024-14Jul2024.csv'),
 PosixPath('data/raw/bikepoint.json'))

## First look

In [3]:
pd.read_csv(journeys_path).head()

,Number,Start date,Start station number,Start station,End date,End station number,End station,Bike number,Bike model,Total duration,Total duration (ms)
0,141082394,2024-07-15 00:00,1036,"Crawford Street, Marylebone",2024-07-15 00:30,300047,"Alfreda Street, Battersea Park",62221.0,PBSC_EBIKE,30m 20s,1820874
1,141082393,2024-07-14 23:59,1066,"Calshot Street , King's Cross",2024-07-15 00:20,3434,"Sancroft Street, Vauxhall",55323.0,CLASSIC,20m 50s,1250126
2,141082392,2024-07-14 23:59,3449,"Breams Buildings, Holborn",2024-07-15 00:16,200020,"Shadwell Station, Shadwell",20895.0,CLASSIC,17m 2s,1022312
3,141082391,2024-07-14 23:59,1143,"Kensington Church Street, Kensington",2024-07-15 00:03,1116,"Emperor's Gate, South Kensington",57732.0,CLASSIC,4m 16s,256109
4,141082390,2024-07-14 23:59,1007,"Drury Lane, Covent Garden",2024-07-15 00:09,1003,"Chapel Place, Marylebone",56159.0,CLASSIC,10m 30s,630150


In [4]:
pd.read_json(bikepoint_path).head()

,$type,id,url,commonName,placeType,additionalProperties,children,childrenUrls,lat,lon
0,"Tfl.Api.Presentation.Entities.Place, Tfl.Api.P...",BikePoints_1,/Place/BikePoints_1,"River Street , Clerkenwell",BikePoint,[{'$type': 'Tfl.Api.Presentation.Entities.Addi...,[],[],51.529163,-0.109970
1,"Tfl.Api.Presentation.Entities.Place, Tfl.Api.P...",BikePoints_2,/Place/BikePoints_2,"Phillimore Gardens, Kensington",BikePoint,[{'$type': 'Tfl.Api.Presentation.Entities.Addi...,[],[],51.499606,-0.197574
2,"Tfl.Api.Presentation.Entities.Place, Tfl.Api.P...",BikePoints_3,/Place/BikePoints_3,"Christopher Street, Liverpool Street",BikePoint,[{'$type': 'Tfl.Api.Presentation.Entities.Addi...,[],[],51.521283,-0.084605
3,"Tfl.Api.Presentation.Entities.Place, Tfl.Api.P...",BikePoints_4,/Place/BikePoints_4,"St. Chad's Street, King's Cross",BikePoint,[{'$type': 'Tfl.Api.Presentation.Entities.Addi...,[],[],51.530059,-0.120973
4,"Tfl.Api.Presentation.Entities.Place, Tfl.Api.P...",BikePoints_5,/Place/BikePoints_5,"Sedding Street, Sloane Square",BikePoint,[{'$type': 'Tfl.Api.Presentation.Entities.Addi...,[],[],51.493130,-0.156876


## Load journeys

`load_journeys` reads a list of journey extracts into one DataFrame. It keeps the columns in `JOURNEY_COLUMNS` under short names, keeps station numbers as strings so their leading zeros match the BikePoint `TerminalName`, and parses the start and end times.

In [5]:
journeys = load_journeys([journeys_path])
journeys.info()

<class 'pandas.DataFrame'>
RangeIndex: 362391 entries, 0 to 362390
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   rental_id       362391 non-null  int64         
 1   bike_id         362390 non-null  Int64         
 2   start           362391 non-null  datetime64[us]
 3   start_terminal  362391 non-null  str           
 4   end             362391 non-null  datetime64[us]
 5   end_terminal    362391 non-null  str           
 6   bike_model      362391 non-null  str           
 7   duration_ms     362391 non-null  int64         
dtypes: Int64(1), datetime64[us](2), int64(2), str(3)
memory usage: 29.2 MB


In [6]:
journeys.head()

,rental_id,bike_id,start,start_terminal,end,end_terminal,bike_model,duration_ms
0,141082394,62221,2024-07-15 00:00:00,001036,2024-07-15 00:30:00,300047,PBSC_EBIKE,1820874
1,141082393,55323,2024-07-14 23:59:00,001066,2024-07-15 00:20:00,003434,CLASSIC,1250126
2,141082392,20895,2024-07-14 23:59:00,003449,2024-07-15 00:16:00,200020,CLASSIC,1022312
3,141082391,57732,2024-07-14 23:59:00,001143,2024-07-15 00:03:00,001116,CLASSIC,256109
4,141082390,56159,2024-07-14 23:59:00,001007,2024-07-15 00:09:00,001003,CLASSIC,630150


### Tests

In [7]:
# Columns are exactly the renamed set
assert list(journeys.columns) == list(JOURNEY_COLUMNS.values())

# Station numbers keep their leading zeros
assert journeys["start_terminal"].str.startswith("0").any()
assert journeys["start_terminal"].str.len().min() == 6

# Start and end times are parsed as datetimes
assert pd.api.types.is_datetime64_any_dtype(journeys["start"])
assert pd.api.types.is_datetime64_any_dtype(journeys["end"])

# Each rental appears once
assert journeys["rental_id"].is_unique

# Passing two files concatenates them
assert len(load_journeys([journeys_path] * 2)) == 2 * len(journeys)

print("All load_journeys tests passed")

All load_journeys tests passed
